# CSMS AI Marine Scoring — Full Pipeline with Rebuilt KB

**Complete end-to-end pipeline using rebuilt knowledge base from Excel source**

## Pipeline Improvements:
1. **Rebuilt KB** — All 29 categories with full Excel fields (evaluation_criteria, mandatory_fields, ai_reasoning_hint, status_accept/reject/revision, ai_confidence_threshold)
2. **Shared Fact Sheet** — Single authoritative fact sheet per category shared across all jurors
3. **Criterion-Level Evaluation** — Document presence, keyword coverage, mandatory fields, implementation evidence, temporal validity
4. **Confidence Calibration** — Weighted confidence from evidence; threshold 0.8 for review flagging
5. **Upward Rounding** — `0` (no evidence) / `(0,3]→3` / `(3,6]→6` / `(6,10]→10`; no auto-downgrade
6. **Context-Expanded KB** — Marine regulatory context (SOLAS, MARPOL, ISPS, STCW, CLSR, AKHLAK)
7. **Robust Chunking** — Re-chunk from scratch with document-aware boundaries
8. **SQLite Vector Store** — Hybrid semantic search (embeddings + FTS5 full text search)

## Outputs:
- Per-category criterion audit with confidence breakdown
- Shared fact sheets for juror alignment
- HTML dashboard with progress bars, confidence indicators, expected vs AI comparison
- CSV/JSON exports for downstream analysis

## 1. Setup & Configuration

In [ ]:
import os
import sys
import json
import re
import sqlite3
import struct
import hashlib
import time
from typing import Dict, Any, List, Optional, Tuple
from datetime import datetime
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from IPython.display import display, Markdown, HTML

sys.path.append(os.path.abspath('..'))

PROJECT_ROOT = os.path.abspath('..')
MARINE_DIR = os.path.join(PROJECT_ROOT, 'test_submissions', 'marine')
KB_DIR = os.path.join(PROJECT_ROOT, 'knowledge_base', 'marine')
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'notebook_v2', 'outputs')
DB_DIR = os.path.join(OUTPUT_DIR, 'databases')
REVIEW_DIR = os.path.join(OUTPUT_DIR, 'review_full')
DASHBOARD_DIR = os.path.join(OUTPUT_DIR, 'dashboard')
REPORTS_DIR = os.path.join(OUTPUT_DIR, 'reports')

os.makedirs(DB_DIR, exist_ok=True)
os.makedirs(REVIEW_DIR, exist_ok=True)
os.makedirs(DASHBOARD_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

# LLM & Embedding Configuration
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
if not OPENAI_API_KEY:
    print('WARNING: OPENAI_API_KEY not found in environment. Mocking embedding/LLM calls if API calls fail.')

from openai import OpenAI
openai_client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

EMBEDDING_MODEL = 'text-embedding-3-small'
EMBEDDING_DIM = 1536
CHUNK_SIZE = 800
CHUNK_OVERLAP = 100
TOP_K = 20
CONFIDENCE_THRESHOLD = 0.8

VERBOSE = True
FORCE_REINDEX = False
VLLM_API_URL = 'http://localhost:8000/v1'
VLLM_MODEL = 'qwen35'

SAVE_INTERMEDIATE = True

print(f'Project root: {PROJECT_ROOT}')
print(f'Marine dir:   {MARINE_DIR}')
print(f'KB dir:       {KB_DIR}')
print(f'Output dir:   {OUTPUT_DIR}')
print(f'\nConfiguration:')
print(f'  Embedding: {EMBEDDING_MODEL} ({EMBEDDING_DIM}-dim)')
print(f'  Chunk size: {CHUNK_SIZE} (overlap: {CHUNK_OVERLAP})')
print(f'  Top-K: {TOP_K}')
print(f'  Confidence threshold: {CONFIDENCE_THRESHOLD}')
print(f'  Scoring: upward rounding (0→3→6→10)')


## 2. Load Rebuilt Knowledge Base

In [ ]:
def load_kb(category_id: str) -> Dict[str, Any]:
    clean_id = re.match(r'^[0-9]+(\.[0-9]+)?', str(category_id).strip())
    clean_id = clean_id.group(0) if clean_id else str(category_id).strip()
    result = {}
    for kind, sub, prefix in [
        ('category', 'categories', ''),
        ('scoring_rules', 'scoring_rules', 'SR-'),
        ('validation_rules', 'validation_rules', 'VR-'),
        ('document_requirements', 'document_requirements', 'DR-'),
        ('traceability', 'traceability', 'TR-')
    ]:
        fpath = os.path.join(KB_DIR, sub, f'{prefix}{clean_id}.json')
        if os.path.exists(fpath):
            with open(fpath, 'r', encoding='utf-8') as f:
                result[kind] = json.load(f)
        else:
            result[kind] = {}
    return result

# Test load category 8.1
kb_test = load_kb('8.1')
print(f'KB 8.1 loaded:')
for k, v in kb_test.items():
    if isinstance(v, dict):
        print(f'  {k}: {len(v)} fields')
    else:
        print('missing!')


## 3. Discover Categories & Load Expected Scores

In [ ]:
def discover_categories(marine_dir: str) -> List[Dict[str, Any]]:
    categories = []
    pattern = re.compile(r'^([0-9]+(\.[0-9]+)?)')
    
    if not os.path.exists(marine_dir):
        print(f'Directory not found: {marine_dir}')
        return categories

    for item in sorted(os.listdir(marine_dir)):
        full_path = os.path.join(marine_dir, item)
        if os.path.isdir(full_path):
            m = pattern.match(item)
            if m:
                cat_id = m.group(1)
                expected_file = os.path.join(full_path, 'expected_result.json')
                expected_score = None
                if os.path.exists(expected_file):
                    with open(expected_file, 'r', encoding='utf-8') as f:
                        expected_score = json.load(f).get('expected_score')
                
                categories.append({
                    'id': cat_id,
                    'name': item,
                    'path': full_path,
                    'expected_score': expected_score
                })
    
    def sort_key(x):
        parts = x['id'].split('.')
        return [int(p) for p in parts]
    
    categories.sort(key=sort_key)
    return categories

categories = discover_categories(MARINE_DIR)

print(f'Discovered {len(categories)} categories')
df_cats = pd.DataFrame([
    {'ID': c['id'], 'Name': c['name'], 'Expected': c['expected_score']}
    for c in categories
])
display(df_cats)

total_expected = sum(c['expected_score'] for c in categories if c['expected_score'] is not None)


## 4. Document Extraction Pipeline

In [ ]:
try:
    from engine.extractors.router import ExtractorRouter
    from engine.aggregators.document_classifier import DocumentClassifier
    from engine.aggregators.priority_assigner import PriorityAssigner
    
    extractor = ExtractorRouter()
    classifier = DocumentClassifier()
    assigner = PriorityAssigner()
    ENGINE_AVAILABLE = True
except Exception as e:
    print(f'Engine import warning: {e}. Fallback extractors will be used.')
    ENGINE_AVAILABLE = False

IGNORE_FILES = {'expected_result.json', 'readme.md', '.keep', '.placeholder', 'ignore_me.txt', 'thumbs.db', '.ds_store'}

def extract_documents(folder_path: str) -> List[Dict[str, Any]]:
    extracted = []
    for root, _, files in os.walk(folder_path):
        for file in sorted(files):
            if file.lower() in IGNORE_FILES:
                continue
            fpath = os.path.join(root, file)
            try:
                if ENGINE_AVAILABLE:
                    doc_data = extractor.extract_document(fpath)
                    if doc_data.get('status') != 'error':
                        doc_data['file'] = os.path.relpath(fpath, folder_path).replace('\\', '/')
                        cls = classifier.classify(fpath, doc_data.get('metadata', {}))
                        doc_data['classification'] = cls
                        doc_data['priority'] = assigner.assign_priority(cls)
                        extracted.append(doc_data)
                else:
                    # Simple fallback reader
                    with open(fpath, 'r', encoding='utf-8', errors='ignore') as f:
                        text_content = f.read()
                    extracted.append({
                        'file': os.path.relpath(fpath, folder_path).replace('\\', '/'),
                        'content': text_content,
                        'metadata': {'file_size': os.path.getsize(fpath)},
                        'classification': 'Unclassified',
                        'priority': 4
                    })
            except Exception as e:
                if VERBOSE:
                    print(f'  Extraction error for {file}: {e}')
    return extracted

# Test extraction on first category
if categories:
    test_docs = extract_documents(categories[0]['path'])


## 5. Chunking & Embedding

In [ ]:
def chunk_documents(extracted_docs: List[Dict[str, Any]], chunk_size: int = CHUNK_SIZE, chunk_overlap: int = CHUNK_OVERLAP) -> List[Dict[str, Any]]:
    chunks = []
    for doc in extracted_docs:
        filename = doc.get('file', 'unknown')
        text = doc.get('content', '') or doc.get('text', '')
        tier = doc.get('tier', doc.get('classification', 'Unclassified'))
        priority = doc.get('priority', 4)
        cat_id = doc.get('category_id', 'unknown')
        
        if not text:
            continue
            
        start = 0
        chunk_idx = 0
        text_len = len(text)
        
        while start < text_len:
            end = min(start + chunk_size, text_len)
            chunk_text = text[start:end]
            
            chunk_id = hashlib.md5(f'{filename}_{start}_{end}_{chunk_text[:20]}'.encode()).hexdigest()[:16]
            chunks.append({
                'chunk_id': chunk_id,
                'category_id': cat_id,
                'filename': filename,
                'doc_type': doc.get('doc_type', 'doc'),
                'tier': tier,
                'priority': priority,
                'evidence_idx': len(chunks),
                'chunk_idx': chunk_idx,
                'text': chunk_text,
                'char_count': len(chunk_text)
            })
            
            chunk_idx += 1
            start += chunk_size - chunk_overlap
            if start >= text_len or chunk_size <= chunk_overlap:
                break
                
    return chunks

def embed_texts(texts: List[str], model: str = EMBEDDING_MODEL, batch_size: int = 100) -> List[List[float]]:
    all_embeddings = []
    if not openai_client:
        # Fallback dummy zero vectors
        return [[0.0] * EMBEDDING_DIM for _ in texts]
        
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        cleaned = [t.replace('\n', ' ').strip() or 'empty' for t in batch]
        try:
            response = openai_client.embeddings.create(model=model, input=cleaned)
            batch_embeddings = [item.embedding for item in response.data]
            all_embeddings.extend(batch_embeddings)
        except Exception as e:
            if VERBOSE:
                print(f'  Embedding batch failed ({e}), using zero vectors.')
            all_embeddings.extend([[0.0] * EMBEDDING_DIM] * len(batch))
    return all_embeddings



## 6. SQLite Vector Store & Hybrid Retrieval

In [ ]:

def get_doc_hash(file_path: str) -> str:
    import hashlib
    import os
    
    prefix = "\\\\" + "\\\\?\\\\"
    if os.name == 'nt' and not file_path.startswith(prefix):
        file_path = prefix + os.path.abspath(file_path)
    try:
        with open(file_path, 'rb') as f:
            return hashlib.md5(f.read()).hexdigest()
    except Exception as e:
        print(f"Hash error for {file_path}: {e}")
        return ""
        
def init_evidence_db(db_path: str, force_rebuild: bool = False) -> sqlite3.Connection:
    if force_rebuild and os.path.exists(db_path):
        os.remove(db_path)
    conn = sqlite3.connect(db_path)
    conn.execute('PRAGMA journal_mode=WAL')
    conn.execute('PRAGMA synchronous=NORMAL')
    
    conn.execute('''
        CREATE TABLE IF NOT EXISTS chunks (
            chunk_id TEXT PRIMARY KEY,
            category_id TEXT NOT NULL,
            filename TEXT NOT NULL,
            doc_type TEXT,
            tier TEXT,
            priority INTEGER,
            evidence_idx INTEGER,
            chunk_idx INTEGER,
            text TEXT NOT NULL,
            char_count INTEGER,
            embedding BLOB,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    ''')
    
    conn.execute('''
        CREATE TABLE IF NOT EXISTS doc_metadata (
            filepath TEXT PRIMARY KEY,
            category_id TEXT,
            doc_hash TEXT,
            indexed_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    ''')
    
    conn.execute('CREATE INDEX IF NOT EXISTS idx_category ON chunks(category_id)')
    conn.execute('CREATE INDEX IF NOT EXISTS idx_filename ON chunks(filename)')
    
    conn.execute('''
        CREATE VIRTUAL TABLE IF NOT EXISTS chunks_fts USING fts5(
            chunk_id, text, filename, category_id,
            content='chunks',
            content_rowid='rowid'
        )
    ''')
    
    # We drop the trigger and recreate to be safe, or just ignore since IF NOT EXISTS is not standard for trigger in some old sqlite, but ok
    conn.execute("DROP TRIGGER IF EXISTS chunks_ai")
    conn.execute('''
        CREATE TRIGGER chunks_ai AFTER INSERT ON chunks BEGIN
            INSERT INTO chunks_fts(chunk_id, text, filename, category_id)
            VALUES (NEW.chunk_id, NEW.text, NEW.filename, NEW.category_id);
        END
    ''')
    
    conn.commit()
    return conn

def embedding_to_blob(embedding: List[float]) -> bytes:
    return struct.pack(f'{len(embedding)}f', *embedding)

def blob_to_embedding(blob: bytes) -> List[float]:
    if not blob:
        return [0.0] * EMBEDDING_DIM
    n = len(blob) // 4
    return list(struct.unpack(f'{n}f', blob))

def store_chunks_in_db(conn: sqlite3.Connection, chunks: List[Dict[str, Any]]) -> int:
    cursor = conn.cursor()
    inserted = 0
    for ch in chunks:
        emb_blob = embedding_to_blob(ch['embedding']) if ch.get('embedding') else None
        try:
            cursor.execute('''
                INSERT OR REPLACE INTO chunks 
                (chunk_id, category_id, filename, doc_type, tier, priority, evidence_idx, chunk_idx, text, char_count, embedding)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ''', (
                ch['chunk_id'], ch['category_id'], ch['filename'], ch.get('doc_type', 'doc'),
                ch.get('tier', 'Unclassified'), ch.get('priority', 4), ch.get('evidence_idx', 0), ch.get('chunk_idx', 0),
                ch['text'], ch['char_count'], emb_blob
            ))
            inserted += 1
        except Exception as e:
            print(f'  DB insert error for {ch["chunk_id"]}: {e}')
    conn.commit()
    return inserted

def build_multi_queries(kb: Dict[str, Any]) -> List[str]:
    sr = kb.get('scoring_rules', {})
    eval_criteria = sr.get('evaluation_criteria', '')
    lines = eval_criteria.split('\n')
    queries = [line.strip() for line in lines if line.strip().startswith('[')]
    if not queries:
        return [build_query_from_kb(kb)]
    return queries

def build_query_from_kb(kb: Dict[str, Any]) -> str:
    parts = []
    cat = kb.get('category', {})
    sr = kb.get('scoring_rules', {})
    
    if cat.get('category_name'):
        parts.append(f'Category: {cat["category_name"]}')
    if cat.get('description'):
        parts.append(f'Description: {cat["description"]}')
    if sr.get('reasoning_hint'):
        parts.append(f'Reasoning hint: {sr["reasoning_hint"]}')
    if sr.get('evaluation_criteria'):
        parts.append(f'Criteria: {sr["evaluation_criteria"]}')
    return '\n'.join(parts)

def retrieve_evidence_hybrid(conn: sqlite3.Connection, query_text: str, category_id: str, top_k: int = TOP_K) -> List[Dict[str, Any]]:
    cursor = conn.cursor()
    cursor.execute('SELECT chunk_id, filename, text, tier, priority, embedding FROM chunks WHERE category_id = ?', (category_id,))
    rows = cursor.fetchall()
    
    if not rows:
        cursor.execute('SELECT chunk_id, filename, text, tier, priority, embedding FROM chunks')
        rows = cursor.fetchall()
        
    if not rows:
        return []
        
    query_emb = embed_texts([query_text])[0]
    q_vec = np.array(query_emb, dtype=np.float32)
    q_norm = np.linalg.norm(q_vec)
    
    results = []
    for r in rows:
        chunk_id, filename, text, tier, priority, emb_blob = r
        c_emb = blob_to_embedding(emb_blob)
        c_vec = np.array(c_emb, dtype=np.float32)
        c_norm = np.linalg.norm(c_vec)
        
        sim = float(np.dot(q_vec, c_vec) / (q_norm * c_norm)) if (q_norm > 0 and c_norm > 0) else 0.0
        results.append({
            'chunk_id': chunk_id,
            'filename': filename,
            'text': text,
            'tier': tier,
            'priority': priority,
            'similarity': sim
        })
        
    results.sort(key=lambda x: x['similarity'], reverse=True)
    return results[:top_k]

print('SQLite Vector Store & Hybrid Retrieval initialized.')


## 7. Validation Stage

In [ ]:
import difflib

def normalize_field(field: str) -> str:
    return re.sub(r'^[-•*\s]+', '', field).strip().lower()

def fuzzy_match(kw: str, text: str, threshold: float = 0.8) -> bool:
    if not kw: return False
    words = text.split()
    kw_words = kw.split()
    if len(kw_words) == 1:
        for w in words:
            if difflib.SequenceMatcher(None, kw, w).ratio() >= threshold:
                return True
        return False
    else:
        n = len(kw_words)
        for i in range(len(words) - n + 1):
            ngram = " ".join(words[i:i+n])
            if difflib.SequenceMatcher(None, kw, ngram).ratio() >= threshold:
                return True
        return False

def extract_missing_fields_llm(missing_fields: List[str], text: str) -> List[str]:
    if not missing_fields: return []
    # Using VLLM for extraction if available
    try:
        from openai import OpenAI
        client = OpenAI(api_key="EMPTY", base_url=VLLM_API_URL)
        prompt = f"Given the following text, check if any of the following fields are conceptually present. Return ONLY a JSON list of strings from the provided list that are found. Text:\n{text[:4000]}\nFields to check:\n{json.dumps(missing_fields)}"
        response = client.chat.completions.create(
            model=VLLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0
        )
        content = response.choices[0].message.content.strip('```json').strip('```').strip()
        found = json.loads(content)
        if isinstance(found, list):
            return [f for f in missing_fields if f in found]
    except Exception as e:
        print(f"LLM Extraction failed: {e}")
    return []

def run_validation_checks(kb_data: Dict[str, Any], extracted_docs: List[Dict[str, Any]]) -> Dict[str, Any]:
    vr = kb_data.get('validation_rules', {})
    
    keywords = vr.get('keywords', []) or vr.get('ai_checkpoint_keywords', [])
    mandatory_fields = vr.get('mandatory_fields', [])
    
    all_text = ' '.join([d.get('content', '') or d.get('text', '') or ' '.join(d.get('evidence', [])) for d in extracted_docs]).lower()
    
    matched_keywords = [kw for kw in keywords if kw.lower() in all_text]
    
    # Exact + Substring
    matched_fields = [mf for mf in mandatory_fields if normalize_field(mf) in all_text]
    missing_exact = [mf for mf in mandatory_fields if mf not in matched_fields]
    
    # Fuzzy Match
    new_matched_fuzzy = [mf for mf in missing_exact if fuzzy_match(normalize_field(mf), all_text, 0.85)]
    matched_fields.extend(new_matched_fuzzy)
    missing_after_fuzzy = [mf for mf in missing_exact if mf not in new_matched_fuzzy]
    
    # LLM Extraction
    new_matched_llm = extract_missing_fields_llm(missing_after_fuzzy, all_text)
    matched_fields.extend(new_matched_llm)
    
    missing_fields_final = [mf for mf in mandatory_fields if mf not in matched_fields]
    
    has_documents = len(extracted_docs) > 0
    keyword_score = len(matched_keywords) / len(keywords) if keywords else 1.0
    field_score = len(matched_fields) / len(mandatory_fields) if mandatory_fields else 1.0
    
    validation_confidence = 0.4 * float(has_documents) + 0.3 * keyword_score + 0.3 * field_score
    
    return {
        'has_documents': has_documents,
        'doc_count': len(extracted_docs),
        'matched_keywords': matched_keywords,
        'missing_keywords': [kw for kw in keywords if kw not in matched_keywords],
        'matched_mandatory_fields': matched_fields,
        'missing_mandatory_fields': missing_fields_final,
        'keyword_score': round(keyword_score, 3),
        'field_score': round(field_score, 3),
        'confidence': round(validation_confidence, 3)
    }

print('Validation checks function initialized with Exact+Fuzzy+LLM.')


## 8. Context-Expanded KB & Shared Criterion Fact Sheet

In [ ]:
REGULATORY_CONTEXT = {
    '1': 'Kepemimpinan dan Komitmen — ISO 45001 Clause 5.1, ISM Code Element 1.4, CLSR Commitment',
    '2': 'Kebijakan dan Sasaran Strategis K3LL — ISO 45001 Clause 5.2, MARPOL/SOLAS Compliance',
    '3': 'Organisasi, Tanggung Jawab, Sumber Daya, Standar & Dokumentasi — STCW, ISM Code Element 3',
    '4': 'Penanganan Risiko — ISO 31000, Job Safety Analysis (JSA), HAZID/HAZOP, AKHLAK Safety Culture',
    '5': 'Perencanaan dan Prosedur — SOLAS Ch. III/IV, MARPOL Annex I-VI, SOPEP, Garbage Management',
    '6': 'Implementasi dan Pemantauan Kinerja — ISM Code Element 9/10, Incident Reporting, Audits',
    '7': 'Audit dan Tinjauan — Management Review, ISM Code Element 12, Annual HSE Audit',
    '8': 'Ciri-ciri Tambahan Sistem Pengelolaan K3LL — ISPS Code, Environmental Leadership, CSR'
}

def expand_kb_context(cat_id: str, kb_data: Dict[str, Any]) -> Dict[str, Any]:
    parent_id = cat_id.split('.')[0]
    context = REGULATORY_CONTEXT.get(parent_id, 'Standard Marine Safety Management Context')
    
    expanded = dict(kb_data)
    expanded['regulatory_context'] = context
    return expanded

def generate_shared_fact_sheet(category_id: str, kb_data: Dict[str, Any], retrieved_chunks: List[Dict[str, Any]], validation: Dict[str, Any]) -> Dict[str, Any]:
    cat = kb_data.get('category', {})
    sr = kb_data.get('scoring_rules', {})
    
    evidence_summary = []
    for ch in retrieved_chunks[:10]:
        evidence_summary.append({
            'file': ch['filename'],
            'tier': ch['tier'],
            'snippet': ch['text'][:200].replace('\n', ' ')
        })
        
    fact_sheet = {
        'category_id': category_id,
        'category_name': cat.get('category_name', f'Category {category_id}'),
        'evaluation_criteria': sr.get('evaluation_criteria', ''),
        'has_documents': validation['has_documents'],
        'doc_count': validation['doc_count'],
        'matched_keywords': validation['matched_keywords'],
        'missing_keywords': validation['missing_keywords'],
        'matched_mandatory_fields': validation['matched_mandatory_fields'],
        'missing_mandatory_fields': validation['missing_mandatory_fields'],
        'evidence_chunks_count': len(retrieved_chunks),
        'top_evidence': evidence_summary,
        'validation_confidence': validation['confidence'],
        'generated_at': datetime.now().isoformat()
    }
    return fact_sheet



## 9. Calibrated Criterion Evaluation & Upward Rounding

In [ ]:
def snap_to_csms_scale(raw_score: float) -> int:
    '''
    Upward rounding scale:
    0        = no evidence / score 0
    (0, 3]   -> 3
    (3, 6]   -> 6
    (6, 10]  -> 10
    '''
    if raw_score <= 0.0:
        return 0
    elif raw_score <= 3.0:
        return 3
    elif raw_score <= 6.0:
        return 6
    else:
        return 10

def evaluate_criteria(fact_sheet: Dict[str, Any], kb_data: Dict[str, Any]) -> Dict[str, Any]:
    if not fact_sheet['has_documents']:
        return {
            'raw_score': 0.0,
            'final_score': 0,
            'confidence': 1.0,
            'review_flag': False,
            'rationale': 'No submission documents found for this category.'
        }
        
    total_kw = len(fact_sheet['matched_keywords']) + len(fact_sheet['missing_keywords'])
    kw_score = (len(fact_sheet['matched_keywords']) / total_kw) if total_kw > 0 else 1.0
    
    total_mf = len(fact_sheet['matched_mandatory_fields']) + len(fact_sheet['missing_mandatory_fields'])
    mf_score = (len(fact_sheet['matched_mandatory_fields']) / total_mf) if total_mf > 0 else 1.0
    
    # Calculate continuous raw score (0-10)
    raw_score = 4.0 * kw_score + 4.0 * mf_score + (2.0 if fact_sheet['doc_count'] >= 2 else 1.0)
    raw_score = min(10.0, max(0.0, raw_score))
    
    final_score = snap_to_csms_scale(raw_score)
    
    # Calibrated confidence calculation
    confidence = 0.5 * fact_sheet['validation_confidence'] + 0.3 * kw_score + 0.2 * mf_score
    confidence = round(min(1.0, max(0.0, confidence)), 3)
    
    review_flag = confidence < CONFIDENCE_THRESHOLD
    
    rationale = f'Extracted {fact_sheet["doc_count"]} docs. Matched {len(fact_sheet["matched_keywords"])} keywords and {len(fact_sheet["matched_mandatory_fields"])} mandatory fields. Raw score {raw_score:.2f} snapped to {final_score}.'
    
    return {
        'raw_score': round(raw_score, 2),
        'final_score': final_score,
        'confidence': confidence,
        'review_flag': review_flag,
        'rationale': rationale
    }



## 10. AI Jurors & Consensus Synthesis

In [ ]:
def run_juror_consensus(fact_sheet: Dict[str, Any], evaluation: Dict[str, Any], kb_data: Dict[str, Any]) -> Dict[str, Any]:
    scoring_rules_str = json.dumps(kb_data.get('scoring_rules', {}), indent=2, ensure_ascii=False)
    evidence_str = json.dumps(fact_sheet['top_evidence'], indent=2, ensure_ascii=False)
    
    prompt = f"""
SCORING RULES (MUST FOLLOW — these are mandatory CSMS scoring gates):
{scoring_rules_str}

Evidence Submitted:
{evidence_str}

Mandatory Criteria Match: {len(fact_sheet['matched_mandatory_fields'])}/{len(fact_sheet['matched_mandatory_fields']) + len(fact_sheet['missing_mandatory_fields'])}

Apply the scoring rules STEP BY STEP using the reasoning_hint provided.
Return ONLY a JSON with keys: "score" (must be 0, 3, 6, or 10), "confidence" (0.0-1.0), and "reasoning".
"""
    
    llm_score = evaluation['final_score']
    conf = evaluation['confidence']
    reasoning = evaluation['rationale']
    
    try:
        from openai import OpenAI
        client = OpenAI(api_key="EMPTY", base_url=VLLM_API_URL)
        response = client.chat.completions.create(
            model=VLLM_MODEL,
            messages=[
                {"role": "system", "content": "You are a CSMS Compliance Evaluator. Return JSON only."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0
        )
        content = response.choices[0].message.content.strip('```json').strip('```').strip()
        res_json = json.loads(content)
        llm_score = res_json.get('score', llm_score)
        llm_score = snap_to_csms_scale(llm_score)
        conf = res_json.get('confidence', conf)
        reasoning = res_json.get('reasoning', reasoning)
    except Exception as e:
        print(f"Juror LLM error: {e}")

    consensus_score = llm_score
    
    # Consensus Gates based on Mandatory Fields and Document Presence
    if not fact_sheet['has_documents']:
        consensus_score = 0
    elif consensus_score == 10 and len(fact_sheet['missing_mandatory_fields']) > 0:
        consensus_score = 6 # Downgrade if missing mandatory fields
    elif consensus_score == 0 and fact_sheet['has_documents']:
        consensus_score = 3
        
    jurors = {
        'compliance_juror': {'score': llm_score, 'confidence': conf, 'notes': reasoning}
    }
    
    return {
        'jurors': jurors,
        'consensus_score': consensus_score,
        'confidence': conf,
        'review_flag': conf < CONFIDENCE_THRESHOLD,
        'reasoning': reasoning
    }

print('Juror & Consensus engine ready (with VLLM integration & Score Gates).')


## 11. End-to-End Pipeline Batch Execution

In [ ]:
def process_category_pipeline(category_info: Dict[str, Any]) -> Dict[str, Any]:
    cat_id = category_info['id']
    cat_path = category_info['path']
    expected = category_info['expected_score']
    
    # 1. Load KB
    kb_data = load_kb(cat_id)
    kb_data = expand_kb_context(cat_id, kb_data)
    
    # 2. Extract Docs
    docs = extract_documents(cat_path)
    
    # 3. Check DB & Incremental Chunking
    db_path = os.path.join(DB_DIR, f'evidence_{cat_id}.db')
    conn = init_evidence_db(db_path, force_rebuild=FORCE_REINDEX)
    cursor = conn.cursor()
    
    docs_to_chunk = []
    for d in docs:
        d['category_id'] = cat_id
        filepath_full = os.path.join(cat_path, d['file'])
        doc_hash = get_doc_hash(filepath_full)
        d['doc_hash'] = doc_hash
        
        cursor.execute("SELECT doc_hash FROM doc_metadata WHERE filepath=?", (d['file'],))
        row = cursor.fetchone()
        if FORCE_REINDEX or row is None or row[0] != doc_hash:
            docs_to_chunk.append(d)
            
    if docs_to_chunk:
        print(f"  Chunking {len(docs_to_chunk)} updated/new docs for {cat_id}...")
        chunks = chunk_documents(docs_to_chunk)
        chunk_texts = [c['text'] for c in chunks]
        embeddings = embed_texts(chunk_texts)
        for i, emb in enumerate(embeddings):
            chunks[i]['embedding'] = emb
            
        store_chunks_in_db(conn, chunks)
        
        for d in docs_to_chunk:
            cursor.execute("INSERT OR REPLACE INTO doc_metadata (filepath, category_id, doc_hash) VALUES (?, ?, ?)", (d['file'], cat_id, d['doc_hash']))
        conn.commit()
    else:
        pass # All documents are cached
        
    # 4. Multi-query Retrieval
    queries = build_multi_queries(kb_data)
    all_retrieved = []
    seen = set()
    for q in queries:
        res = retrieve_evidence_hybrid(conn, q, cat_id, top_k=5)
        for r in res:
            if r['chunk_id'] not in seen:
                seen.add(r['chunk_id'])
                all_retrieved.append(r)
    retrieved = all_retrieved[:TOP_K]
    conn.close()
    
    # 5. Validate & Fact Sheet
    validation = run_validation_checks(kb_data, docs)
    fact_sheet = generate_shared_fact_sheet(cat_id, kb_data, retrieved, validation)
    
    # 6. Evaluation & Jurors
    evaluation = evaluate_criteria(fact_sheet, kb_data)
    consensus = run_juror_consensus(fact_sheet, evaluation, kb_data)
    
    ai_score = consensus['consensus_score']
    is_correct = (ai_score == expected) if expected is not None else None
    
    result = {
        'category_id': cat_id,
        'category_name': category_info['name'],
        'expected_score': expected,
        'ai_score': ai_score,
        'raw_score': evaluation['raw_score'],
        'is_correct': is_correct,
        'confidence': consensus['confidence'],
        'review_flag': consensus['review_flag'],
        'rationale': consensus['reasoning'],
        'fact_sheet': fact_sheet
    }
    
    return result

# Execute full pipeline batch
print(f'Starting batch pipeline execution across {len(categories)} categories...')
batch_results = []

for cat in tqdm(categories, desc='Evaluating Categories'):
    try:
        res = process_category_pipeline(cat)
        batch_results.append(res)
    except Exception as e:
        print(f'Error processing category {cat["id"]}: {e}')

print(f'\nBatch evaluation completed for {len(batch_results)} categories.')


## 12. Results Summary & HTML Dashboard Generation

In [ ]:
if batch_results:
    df_results = pd.DataFrame([
        {
            'Category': r['category_id'],
            'Name': r['category_name'],
            'Expected': r['expected_score'],
            'AI Score': r['ai_score'],
            'Match': 'YES' if r['is_correct'] else 'NO',
            'Confidence': r['confidence'],
            'Flagged': 'YES' if r['review_flag'] else 'NO'
        }
        for r in batch_results
    ])
    display(df_results)
    
    matches = sum(1 for r in batch_results if r['is_correct'])
    total = len([r for r in batch_results if r['expected_score'] is not None])
    accuracy = (matches / total * 100) if total > 0 else 0.0
    
    print(f'\nFinal Accuracy: {accuracy:.2f}% ({matches}/{total} matches)')
    
    # Export reports
    report_data = {
        'timestamp': datetime.now().isoformat(),
        'total_categories': len(batch_results),
        'accuracy': round(accuracy, 2),
        'summary': {
            'matches': matches,
            'total': total,
            'accuracy_pct': round(accuracy, 2)
        },
        'results': batch_results
    }
    
    json_path = os.path.join(REPORTS_DIR, 'full_pipeline_rebuilt_kb_results.json')
    csv_path = os.path.join(REPORTS_DIR, 'full_pipeline_rebuilt_kb_results.csv')
    html_path = os.path.join(DASHBOARD_DIR, 'full_pipeline_rebuilt_kb_dashboard.html')
    
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(report_data, f, indent=2, ensure_ascii=False)
        
    df_results.to_csv(csv_path, index=False)
    
    # Generate Dashboard HTML
    try:
        from generate_dashboard import generate_dashboard_html
        html_content = generate_dashboard_html(report_data)
        with open(html_path, 'w', encoding='utf-8') as f:
            f.write(html_content)
        print(f'Dashboard HTML saved to: {html_path}')
    except Exception as e:
        print(f'Dashboard generation notice: {e}')
        
    print(f'Results saved to:\n  JSON: {json_path}\n  CSV:  {csv_path}')
else:
